# 04_step3_RCM_BA_F2 - Calibration of F2: RCM 2025 no-hosing -> target GCM ERA5-like 0.25 deg (RCM domain)

**F2 = fixed WDF + QDM** (per month and per pixel):

- **target** (`ref`): `GCM_2025_nohosing_ERA5like_0.25deg` (output of notebook step2);
- **to correct** (`hist`): `RCM_2025_nohosing` (0.25 deg), on its own domain.

The global target is cropped to the RCM domain (plus a small margin) and put on the RCM grid: there is no hand-made region box.

Why this pair: the RCM is nested in EC-Earth3, so both share the same large-scale climate (same fixed 2025 forcing).
The correction should therefore only act on the RCM's own bias and on small-scale detail.

| variable | treatment |
|---|---|
| `pr` | fixed WDF (dry days), then multiplicative QDM on wet days |
| `tasmax` | additive QDM |
| `tasmin` | through the diurnal range `DTR = tasmax - tasmin` (multiplicative QDM); `tasmin = adjusted tasmax - adjusted DTR` |

The WDF rule is estimated here **once** and applied unchanged to all RCM runs (step4): the dry-day frequency
of the target is never re-imposed on a hosing run. No correction is calibrated on the hosing runs.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

RCM_REF_RUN = "rcm_2025_nohosing"
TARGET_FILE = "gcm_2025_nohosing_era5like_025.nc"
CHUNKS_SPACE = {"lat": 30, "lon": 60}
ALLOW_TARGET_REGRID = True      # if the RCM grid is not a subset of the target grid: regrid the target onto the RCM grid
TARGET_MARGIN = 1.0             # degrees kept around the RCM domain when cropping the global target
ERA5LIKE_DIR = bu.out_dir(cfg, "plan_B", "era5like_025")
F2_DIR = bu.out_dir(cfg, "plan_B", "F2_BA")

## 1. Loading and grid alignment

In [ ]:
target_path = ERA5LIKE_DIR / TARGET_FILE
assert target_path.exists(), "Target not found: run notebook step2 first."
rcm = {v: bu.read_raw(cfg, RCM_REF_RUN, v) for v in VARS}      # the RCM keeps its own domain
dom = bu.domain_of(rcm["pr"], TARGET_MARGIN)                    # RCM domain (+ margin), no hand-made box
target = {v: bu.crop_bbox(bu.read_processed(target_path, v), dom) for v in VARS}

for v in VARS:
    try:
        target[v] = bu.sub_to(target[v], rcm[v])     # RCM grid = subset of the target grid: exact selection
    except KeyError:
        msg = f"{v}: the RCM grid ({rcm[v].sizes['lat']}x{rcm[v].sizes['lon']}) is not a subset of the target grid"
        if not ALLOW_TARGET_REGRID:
            raise ValueError(msg)
        print(msg, "-> the target is regridded (conservative) onto the RCM grid")
        target[v] = bu.regrid_conservative(target[v], rcm[v].lat.values, rcm[v].lon.values)
    assert bu.same_grid(target[v], rcm[v])
    print(f"{v:7s} target {target[v].sizes['time']} days | RCM {rcm[v].sizes['time']} days | "
          f"grid {rcm[v].sizes['lat']}x{rcm[v].sizes['lon']}")

## 2. Calibration of F2

In [ ]:
bu.fit_F(target, rcm, F2_DIR, chunks_space=CHUNKS_SPACE)
print(sorted(p.name for p in F2_DIR.iterdir()))

## 3. WDF rule of the RCM

`p0_ref` / `p0_hist`: dry-day fraction of the target / of the RCM; `t`: threshold applied to the RCM (above 0.1 mm if the RCM is too wet);
`q`: probability of activating an RCM dry day (if the RCM is too dry).

In [ ]:
par = xr.open_dataset(F2_DIR / "wdf_params.nc").load()
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for r, m in enumerate([1, 7]):
    for c, (k, kw) in enumerate([("p0_ref", dict(vmin=0, vmax=1)), ("p0_hist", dict(vmin=0, vmax=1)),
                                 ("t", dict(vmin=0.1, vmax=1.0)), ("q", dict(vmin=0, vmax=0.5))]):
        a = par[k].sel(month=m)
        im = axes[r, c].pcolormesh(a.lon, a.lat, a, shading="auto", **kw); plt.colorbar(im, ax=axes[r, c], shrink=0.7)
        axes[r, c].set_title(f"{k} - month {m}  (ref = GCM target, hist = RCM)")
plt.tight_layout(); plt.savefig(F2_DIR / "wdf_rule.png", dpi=110); plt.show()

## 4. In-sample check

F2 is applied to the RCM 2025 no-hosing run itself: the quantiles should fall back onto those of the target.
(Code check only; the two-halves validation is in the validations notebook.)

In [ ]:
sample_idx = slice(0, 365 * 10)
Hs = {v: x.isel(time=sample_idx).compute() for v, x in rcm.items()}
Ts = {v: x.isel(time=sample_idx).compute() for v, x in target.items()}
As = xr.Dataset(bu.apply_F(Hs, F2_DIR, chunks_space=CHUNKS_SPACE)).compute()   # single computation
rows = []
for v in VARS:
    for lab, x in (("GCM ERA5-like target", Ts[v]), ("raw RCM", Hs[v]), ("RCM adjusted F2", As[v])):
        s = bu.summarize(x)
        rows.append({"var": v, "series": lab, **{k: round(val, 3) for k, val in s.items() if k != "n"}})
pd.DataFrame(rows)